# Aprendizado de Máquina — Aula prática 05

## Taxas de convergência e a maldição da dimensionalidade

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

Esta aula prática é diferente das outras: **não há um conjunto de dados para
analisar**. O objeto de estudo aqui é a própria teoria. A Aula 04 terminou com uma
figura incômoda — o KNN, que ganhava com folga em uma dimensão, perdia para uma
regressão linear *errada* quando havia 20 covariáveis. Aquilo não foi azar de
semente nem detalhe de implementação:

> **é uma consequência de geometria, e dá para medir cada passo do argumento.**

O plano é medir, em ordem: o raio que reúne $k$ vizinhos; a distância ao vizinho
mais próximo conforme $d$ cresce; a taxa $n^{-2/(2+d)}$ com que o risco cai; e o
tamanho de amostra que ela exige. Nos últimos terços, as duas rotas de fuga —
esparsidade e redundância — e por que elas explicam tanto o fracasso do KNN nas
resenhas da Amazon quanto o sucesso dele no `superconductivity.csv`.

### Objetivos

Ao final deste notebook você deve ser capaz de:

- medir o raio $\rho \approx (k/n)^{1/d}$ e recuperar o expoente $1/d$ dos dados;
- mostrar que, em dimensão alta, todos os pontos ficam a distâncias parecidas;
- estimar empiricamente o expoente da taxa de convergência do KNN e confrontá-lo
  com $n^{-2/(2+d)}$ e $n^{-4/(4+d)}$;
- traduzir uma taxa em tamanho de amostra necessário;
- distinguir **esparsidade** de **redundância** e reconhecer qual método se salva
  em cada caso;
- estimar a **dimensão intrínseca** de um conjunto de dados real.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
import pandas as pd
from matplotlib.pyplot import subplots

O objeto novo é o `NearestNeighbors`, que devolve as distâncias aos vizinhos sem
ajustar modelo nenhum — é a ferramenta certa para estudar a geometria dos dados.
A função gama entra uma vez só, na Seção 2, para calcular o volume de uma bola em
$\R^d$.

In [ ]:
import sklearn.linear_model as skl
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors
from sklearn.preprocessing import StandardScaler

from scipy.special import gamma

In [ ]:
import warnings
warnings.filterwarnings("ignore")

---
## 2. O raio que reúne $k$ vizinhos

Todo o resto da aula sai de uma conta de volume. Numa bola de raio $\rho$ em
$\R^d$, o volume cresce como $\rho^d$. Se $n$ pontos estão espalhados num conjunto
de volume 1, uma bola de raio $\rho$ contém em média $\approx n\rho^d$ deles. Para
que ela contenha os $k$ vizinhos que queremos:

$$n\rho^d \approx k \qquad\Longrightarrow\qquad \rho \approx \left(\frac{k}{n}\right)^{1/d}.$$

O expoente $1/d$ é o vilão de toda a aula. Vamos ver se ele aparece de fato.

In [ ]:
def raio_kesimo(n, d, k, rng, n_alvos=300):
    """Distancia media ao k-esimo vizinho, em [0,1]^d."""
    X = rng.uniform(0, 1, size=(n, d))
    alvos = rng.uniform(0, 1, size=(n_alvos, d))
    dist, _ = NearestNeighbors(n_neighbors=k).fit(X).kneighbors(alvos)
    return dist[:, -1].mean()


rng = np.random.default_rng(4)
n, k = 10_000, 10
tabela = []
for d in [1, 2, 5, 10, 20]:
    V_d = np.pi ** (d / 2) / gamma(d / 2 + 1)      # volume da bola de raio 1
    tabela.append({"d": d,
                   "(k/n)^(1/d)": (k / n) ** (1 / d),
                   "(k/(n V_d))^(1/d)": (k / (n * V_d)) ** (1 / d),
                   "raio medido": raio_kesimo(n, d, k, rng)})
pd.DataFrame(tabela).set_index("d").round(4)

A versão simplificada acerta a ordem de grandeza e conta a história certa: em
$d=1$ os dez vizinhos cabem num intervalo minúsculo — são **de fato** vizinhos.
Em $d=20$, para reunir dez pontos você precisa de uma bola de raio **maior que 1**,
isto é, maior que o lado do cubo inteiro. A palavra "vizinho" perdeu o sentido, e
com ela a justificativa inteira do método.

A coluna do meio mostra de onde vem a diferença. A conta honesta não é $n\rho^d=k$
e sim $n V_d \rho^d = k$, onde $V_d = \pi^{d/2}/\Gamma(d/2+1)$ é o volume da bola
de raio 1. Com ela, a previsão bate **exatamente** em $d=1$ e $d=2$. Daí em diante
a previsão fica abaixo do medido por um motivo que a Seção 4 vai reencontrar: uma
fração enorme dos alvos está perto da borda do cubo, onde metade da bola cai fora e
sobram menos pontos para pegar.

O que nos interessa, porém, é o **expoente**, e esse não depende de constante
nenhuma: basta variar $k/n$ e olhar a inclinação em escala log–log.

In [ ]:
razoes = np.array([1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2])
n_fixo = 20_000

fig, ax = subplots(figsize=(5.2, 3.2))
print(f"{'d':>3} {'inclinacao medida':>18} {'1/d':>8} {'k/n para o raio cair a metade':>32}")
for d in [2, 5, 10, 20]:
    raios = [raio_kesimo(n_fixo, d, max(1, int(round(rho * n_fixo))), rng)
             for rho in razoes]
    inclinacao = np.polyfit(np.log(razoes), np.log(raios), 1)[0]
    fator = 2 ** (1 / inclinacao)
    print(f"{d:3d} {inclinacao:18.4f} {1/d:8.4f} {'dividir por ' + f'{fator:.3g}':>32}")
    ax.plot(razoes, raios, "o-", ms=4, label=f"d = {d}")

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("k / n"); ax.set_ylabel("raio ate o k-esimo vizinho")
ax.set_title("a inclinacao em log-log e 1/d", fontsize=9)
ax.legend(fontsize=8)

As inclinações medidas seguem $1/d$ de perto até $d=10$ e ficam um pouco acima
dele em $d=20$ — de novo o efeito de borda, que fica mais forte quanto maior a
dimensão. Mas o que importa é a última coluna. Ela responde: *para conseguir
vizinhos duas vezes mais próximos, quantas vezes mais dados eu preciso?*

Em $d=2$ a resposta é "quatro vezes", o que é razoável. Em $d=20$ é da ordem de
dezenas de milhares — e isso mantendo $k$ fixo. Não existe orçamento de coleta que
compre proximidade em dimensão alta.

> **Sua vez.** Meça o raio até o $k$-ésimo vizinho quando os alvos são pontos do
> **próprio** conjunto (em vez de pontos novos sorteados). O raio fica maior ou
> menor? Pense no que muda: um ponto do conjunto é vizinho de si mesmo, e além
> disso ele está onde os dados estão, e não onde a distribuição é rala.

---
## 3. Em dimensão alta, todo mundo está longe — e à mesma distância

A consequência mais desconcertante da conta anterior não é o vizinho ficar longe:
é **todos** ficarem a distâncias parecidas. Se o mais próximo e o mais distante
estão praticamente à mesma distância, "os $k$ mais próximos" vira uma seleção
quase arbitrária.

In [ ]:
dims = [1, 2, 3, 5, 10, 20, 50, 100]
n_g = 1000
rng_g = np.random.default_rng(4)

medias, p05, contraste = [], [], []
for d in dims:
    X = rng_g.uniform(0, 1, size=(n_g, d))
    alvos = rng_g.uniform(0, 1, size=(200, d))
    dist = np.sqrt(((alvos[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))
    perto, longe = dist.min(axis=1), dist.max(axis=1)
    medias.append((perto / np.sqrt(d)).mean())       # fracao do diametro do cubo
    p05.append(np.quantile(perto / np.sqrt(d), 0.05))
    contraste.append(((longe - perto) / perto).mean())

resumo = pd.DataFrame({"d": dims,
                       "dist. ao +proximo / diametro": medias,
                       "(mais longe - mais perto)/mais perto": contraste}).set_index("d")
resumo.round(3)

In [ ]:
fig, (ax1, ax2) = subplots(1, 2, figsize=(7.6, 3.0))
ax1.plot(dims, medias, "o-", ms=4, color="crimson", label="media")
ax1.fill_between(dims, p05, medias, color="crimson", alpha=0.15,
                 label="entre o percentil 5 e a media")
ax1.set_xscale("log"); ax1.set_xticks(dims); ax1.set_xticklabels(dims)
ax1.set_xlabel("d (escala log)")
ax1.set_ylabel("dist. ao vizinho mais proximo\n(fracao do diametro do cubo)")
ax1.legend(fontsize=7.5)

ax2.plot(dims, contraste, "s-", ms=4, color="steelblue")
ax2.set_xscale("log"); ax2.set_yscale("log")
ax2.set_xticks(dims); ax2.set_xticklabels(dims)
ax2.set_xlabel("d (escala log)")
ax2.set_ylabel("contraste de distancias")
ax2.set_title("o mais longe deixa de ser mais longe", fontsize=9)

Duas leituras. À esquerda: com $n=1000$ pontos, em $d=1$ o vizinho mais próximo
está praticamente colado; em $d=10$ já está a 16% do diâmetro do cubo; em
$d=100$, a um terço.

À direita, a versão que mata o método: o **contraste** entre a maior e a menor
distância desaba. Em $d=1$ o ponto mais distante está mais de dez mil vezes mais
longe que o mais próximo; em $d=100$, está a menos de uma vez e meia. Quando todas
as distâncias são parecidas, ordenar por distância é quase sortear.

---
## 4. A taxa de convergência, medida

O Teorema da aula diz que o risco do KNN é limitado por

$$\underbrace{K\left(\frac{k}{n}\right)^{2/d}}_{\text{viés}^2} + \underbrace{\frac{\sigma^2}{k}}_{\text{variância}},$$

e que, escolhendo $k \asymp n^{2/(2+d)}$ para equilibrar os dois, o risco cai como
$n^{-2/(2+d)}$. Se além de Lipschitz a função for duas vezes diferenciável, o viés
da média local é de segunda ordem e a taxa melhora para $n^{-4/(4+d)}$.

Vamos medir o expoente. A população é a mesma em toda dimensão, com todas as
coordenadas importando igualmente:

$$X \sim \mathrm{Unif}[0,1]^d, \qquad
  r(x) = \frac{1}{\sqrt d}\sum_{j=1}^d \sin(2\pi x_j), \qquad
  \sigma = 0{,}3 .$$

A normalização por $\sqrt d$ não é cosmética: sem ela, a variância de $r$ cresceria
com $d$ e o problema mudaria de dificuldade junto com a dimensão, contaminando a
medida. Assim, $\Var(r(X)) = 1/2$ em qualquer $d$.

In [ ]:
SIGMA = 0.3


def r_dim(X):
    return np.sin(2 * np.pi * X).sum(axis=1) / np.sqrt(X.shape[1])


rng_t = np.random.default_rng(7)
print("Var(r(X)) por dimensao:",
      {d: round(float(r_dim(rng_t.uniform(0, 1, (50_000, d))).var()), 4)
       for d in [1, 2, 4, 8]})

Para cada $(d, n)$ ajustamos o KNN com uma grade de $k$ e ficamos com o **melhor**
— é o $k$ que a teoria escolheria conhecendo tudo. Assim medimos a taxa do método,
sem misturar nela o erro de escolher $k$.

E, como sempre nesta disciplina, avaliamos contra $r$ e não contra $Y$: o risco é
$\mathbb{E}[(\widehat r - r)^2] + \sigma^2$, e o $\sigma^2$ é uma constante que não
participa da taxa. Deixá-lo de fora deixa o expoente visível.

In [ ]:
ns = np.array([100, 300, 1000, 3000, 10_000])
dims_t = [1, 2, 4, 8]
rng_t = np.random.default_rng(7)

curvas, inclinacoes = {}, {}
for d in dims_t:
    Xte = rng_t.uniform(0, 1, size=(3000, d))
    r_te = r_dim(Xte)
    riscos, ks_otimos = [], []
    for n in ns:
        Xtr = rng_t.uniform(0, 1, size=(n, d))
        ytr = r_dim(Xtr) + rng_t.normal(0, SIGMA, size=n)
        ks = np.unique(np.round(np.logspace(0, np.log10(n / 4), 14)).astype(int))
        erros = [np.mean((KNeighborsRegressor(n_neighbors=int(k))
                          .fit(Xtr, ytr).predict(Xte) - r_te) ** 2) for k in ks]
        riscos.append(min(erros))
        ks_otimos.append(int(ks[int(np.argmin(erros))]))
    curvas[d] = (np.array(riscos), ks_otimos)
    inclinacoes[d] = np.polyfit(np.log(ns), np.log(curvas[d][0]), 1)[0]

pd.DataFrame({
    "d": dims_t,
    "inclinacao medida": [inclinacoes[d] for d in dims_t],
    "-2/(2+d)  (Lipschitz)": [-2 / (2 + d) for d in dims_t],
    "-4/(4+d)  (2x derivavel)": [-4 / (4 + d) for d in dims_t],
    "k* em n=10000": [curvas[d][1][-1] for d in dims_t],
}).set_index("d").round(3)

In [ ]:
fig, ax = subplots(figsize=(5.4, 3.4))
for d in dims_t:
    riscos, _ = curvas[d]
    linha, = ax.plot(ns, riscos, "o-", ms=4, label=f"d = {d}  (medido {inclinacoes[d]:+.2f})")
    ref = riscos[0] * (ns / ns[0]) ** (-4 / (4 + d))
    ax.plot(ns, ref, ls="--", lw=1, color=linha.get_color())
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("n"); ax.set_ylabel("risco (sem o sigma^2)")
ax.set_title("cheio: medido   tracejado: n^{-4/(4+d)}", fontsize=9)
ax.legend(fontsize=7.5)

> **A lição.** As inclinações medidas caem **entre** as duas previsões teóricas, e
> a distância até a mais otimista cresce com $d$: em $d=1$ e $d=2$ o decaimento é
> praticamente o de uma função duas vezes diferenciável; em $d=8$ ele fica bem
> aquém disso, mais perto do limitante pessimista de Lipschitz.
>
> Faz sentido, e o motivo é a fronteira. A taxa rápida vale quando a vizinhança de
> um ponto é aproximadamente simétrica em torno dele, e os desvios de $r$ se
> cancelam. Isso vale no miolo do cubo — mas a fração do cubo que é "miolo" é
> $(1-2\rho)^d$, e com $\rho$ da ordem de $0{,}3$ e $d=8$ isso é menos de $1\%$.
> Em dimensão alta **quase tudo é fronteira**, e é o viés de fronteira da Aula 04,
> multiplicado, que aparece aqui.
>
> Para o que interessa, tanto faz qual das duas curvas: as duas achatam com $d$, e
> é isso que a próxima seção traduz em dinheiro.

> **Sua vez.** Refaça a medição com $\sigma = 0{,}05$ em vez de $0{,}3$. A taxa
> (a inclinação) muda? E o $k^\*$ ótimo? Reveja a fórmula
> $K(k/n)^{2/d} + \sigma^2/k$ antes de rodar e faça uma previsão.

---
## 5. Abrindo os dois termos do teorema

A taxa saiu de equilibrar duas parcelas:

$$\underbrace{K\left(\frac{k}{n}\right)^{2/d}}_{\text{viés}^2,\ \text{cresce com }k}
  \;+\;
  \underbrace{\frac{\sigma^2}{k}}_{\text{variância},\ \text{decresce com }k}.$$

As duas são mensuráveis separadamente, repetindo a simulação: o viés é a distância
entre a **estimativa média** e $r$, e a variância é a dispersão das estimativas em
torno dessa média. Vamos ver os dois termos se cruzando, em $d=1$ e em $d=4$.

In [ ]:
def decompor(d, n, ks, n_rep=120, semente=31, n_teste=1500, design_fixo=False):
    """Vies^2 e variancia do KNN, por simulacao.

    Com design_fixo=True as covariaveis de treino sao as MESMAS em todas as
    repeticoes, e so o ruido e resorteado -- e a variancia condicional ao desenho,
    que e a do teorema.
    """
    rng = np.random.default_rng(semente)
    Xte = rng.uniform(0, 1, size=(n_teste, d))
    r_te = r_dim(Xte)
    Xtr_fixo = rng.uniform(0, 1, size=(n, d))

    preds = np.zeros((len(ks), n_rep, n_teste))
    for b in range(n_rep):
        Xtr = Xtr_fixo if design_fixo else rng.uniform(0, 1, size=(n, d))
        ytr = r_dim(Xtr) + rng.normal(0, SIGMA, size=n)
        for j, k in enumerate(ks):
            preds[j, b] = KNeighborsRegressor(n_neighbors=int(k)).fit(Xtr, ytr).predict(Xte)

    media = preds.mean(axis=1)                       # E_D[rhat(x)]
    vies2 = ((media - r_te) ** 2).mean(axis=1)
    variancia = preds.var(axis=1).mean(axis=1)
    return vies2, variancia


n_dec = 2000
ks_dec = np.unique(np.round(np.logspace(0, np.log10(n_dec / 4), 12)).astype(int))
dec = {d: decompor(d, n_dec, ks_dec) for d in (1, 4)}
dec_fixo = {d: decompor(d, n_dec, ks_dec, design_fixo=True) for d in (1, 4)}

In [ ]:
fig, axes = subplots(1, 2, figsize=(7.8, 3.2), sharey=True)
for ax, d in zip(axes, (1, 4)):
    vies2, variancia = dec[d]
    ax.plot(ks_dec, vies2, "o-", ms=3.5, color="steelblue", label="vies^2")
    ax.plot(ks_dec, variancia, "s-", ms=3.5, color="crimson", label="variancia")
    ax.plot(ks_dec, vies2 + variancia, "^-", ms=3.5, color="black", label="soma")
    ax.plot(ks_dec, SIGMA ** 2 / ks_dec, ls="--", lw=1, color="gray",
            label="sigma^2 / k")
    k_min = ks_dec[np.argmin(vies2 + variancia)]
    ax.axvline(k_min, ls=":", color="green")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel("k"); ax.set_title(f"d = {d}   (k* medido = {k_min})", fontsize=9)
axes[0].set_ylabel("contribuicao ao risco"); axes[0].legend(fontsize=7.5)

print(f"{'d':>2} {'k* medido':>10} {'n^(2/(2+d))':>13}"
      f" {'var x k (desenho novo)':>23} {'var x k (desenho fixo)':>23} {'sigma^2':>9}")
for d in (1, 4):
    vies2, variancia = dec[d]
    _, var_fixo = dec_fixo[d]
    j = int(np.argmin(vies2 + variancia))
    print(f"{d:2d} {ks_dec[j]:10d} {n_dec ** (2 / (2 + d)):13.1f}"
          f" {variancia[j] * ks_dec[j]:23.4f} {var_fixo[j] * ks_dec[j]:23.4f}"
          f" {SIGMA ** 2:9.4f}")

Três coisas para conferir na figura e na tabela.

**O viés cresce com $k$**, porque vizinhos mais numerosos são vizinhos mais
distantes — é o Lema da Seção 2 entrando na conta.

**O mínimo da soma está onde as duas curvas se cruzam**, a menos de constantes. Foi
esse o truque da demonstração, e o $k^*$ medido fica na ordem de grandeza de
$n^{2/(2+d)}$. Note como o $k^*$ despenca quando $d$ sobe: em dimensão alta cada
vizinho a mais custa caro demais em viés, e o método é empurrado para médias de
poucos pontos — isto é, para a variância alta. Não há saída boa.

**A variância é $\sigma^2/k$ — mas condicionada ao desenho.** Aqui vale ler as
duas últimas colunas com cuidado. Com o desenho **fixo**, `var x k` devolve
$\sigma^2$ nas duas dimensões, e a demonstração é imediata: dadas as posições dos
pontos, a predição é a média de $k$ respostas independentes de variância
$\sigma^2$. Com um desenho **novo** a cada repetição, o número fica acima —
discretamente em $d=1$, pelo dobro em $d=4$.

A diferença é uma segunda fonte de variabilidade que o teorema não contabiliza:
**quem são os vizinhos** também muda de amostra para amostra. Em $d=1$ com
$k \approx 90$, a composição da vizinhança é estável e isso quase não pesa. Em
$d=4$ com $k = 10$, cada nova amostra escolhe dez pontos bem diferentes, e a
predição balança junto. É mais um item na conta que a dimensão cobra — e um bom
lembrete de que "variância" só quer dizer alguma coisa quando se diz **em relação
a que sorteio**.

---
## 6. A tradução: quantos dados seriam necessários

Uma inclinação em gráfico log–log é abstrata. A pergunta prática é: **quantas
observações eu preciso para atingir um risco alvo?** Invertendo
$\text{risco} \approx n^{-2/(2+d)}$, obtemos $n \approx \delta^{-(2+d)/2}$ — um
crescimento exponencial em $d$.

In [ ]:
alvo = 0.05
dd = np.arange(1, 21)
preciso = alvo ** (-(2 + dd) / 2)

fig, (ax1, ax2) = subplots(1, 2, figsize=(7.6, 3.0))
for d in [1, 2, 5, 10, 20]:
    nn = np.logspace(1, 6, 200)
    ax1.plot(nn, nn ** (-2 / (2 + d)), label=f"d = {d}")
ax1.set_xscale("log"); ax1.set_yscale("log")
ax1.set_xlabel("n"); ax1.set_ylabel("risco proporcional a n^(-2/(2+d))")
ax1.set_title("a taxa achata conforme d cresce", fontsize=9)
ax1.legend(fontsize=7.5)

ax2.plot(dd, preciso, "o-", ms=3, color="crimson")
ax2.axhline(8.1e9, ls=":", color="gray", lw=1)
ax2.text(1.2, 1.3e10, "populacao da Terra", fontsize=7, color="gray")
ax2.set_yscale("log"); ax2.set_xticks([1, 5, 10, 15, 20])
ax2.set_xlabel("d"); ax2.set_ylabel(f"n para atingir risco {alvo}")
ax2.set_title("e o n exigido explode", fontsize=9)

for d in [1, 2, 5, 10, 20]:
    print(f"d = {d:2d}  ->  n = {alvo ** (-(2 + d) / 2):.3g}")
print(f"\nem d=20, isso e {alvo ** (-22 / 2) / 8.1e9:.0f} vezes a populacao da Terra")

Em $d=1$ bastam 90 observações. Em $d=10$ já são dezenas de milhões. Em $d=20$, o
número é de outra natureza: nem coletando um registro de cada ser humano vivo,
milhares de vezes, você chegaria lá.

E aqui vale lembrar o que a aula chama de **taxa minimax**: essa taxa não é a
fraqueza de um método, é o melhor que *qualquer* estimador garante no pior caso da
classe de funções suaves. Trocar o KNN por outra coisa não resolve. A saída tem de
vir de uma **suposição adicional** sobre o problema — e é o resto do notebook.

---
## 7. Fuga 1: esparsidade

A hipótese de **esparsidade** diz que $r$ depende só de um punhado $S$ de
covariáveis, $|S| = s \ll d$. Há $d$ colunas, mas só $s$ importam.

Este é o caso das resenhas da Amazon com que a aula abre: cada covariável é a
contagem de uma palavra, e quase nenhuma palavra diz algo sobre a nota. Vamos
montar exatamente essa situação em miniatura: $d = 20$ colunas, das quais **duas**
carregam todo o sinal.

In [ ]:
def r_esparsa(X):
    return np.sin(2 * np.pi * X[:, 0]) + np.sin(2 * np.pi * X[:, 1])


rng_e = np.random.default_rng(15)
n_e, d_e = 2000, 20
X_tr = rng_e.uniform(0, 1, size=(n_e, d_e))
y_tr = r_esparsa(X_tr) + rng_e.normal(0, SIGMA, size=n_e)
X_te = rng_e.uniform(0, 1, size=(3000, d_e))
r_te = r_esparsa(X_te)

print(f"risco de quem chuta a media: {r_te.var():.4f}")

In [ ]:
def risco_de(modelo, colunas=None):
    a = X_tr if colunas is None else X_tr[:, colunas]
    b = X_te if colunas is None else X_te[:, colunas]
    return np.mean((modelo.fit(a, y_tr).predict(b) - r_te) ** 2)


floresta = RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)

linhas = [
    ("KNN com as 20 colunas", risco_de(KNeighborsRegressor(10))),
    ("KNN so com as 2 relevantes", risco_de(KNeighborsRegressor(10), [0, 1])),
    ("floresta aleatoria, 20 colunas", risco_de(floresta)),
    ("regressao linear, 20 colunas", risco_de(skl.LinearRegression())),
]
pd.DataFrame(linhas, columns=["metodo", "risco"]).set_index("metodo").round(4)

Com as 20 colunas, o KNN corta menos da metade do risco de quem chuta a média. Com
**as duas colunas certas**, corta 99% — o risco cai por um fator de quarenta. As 18
colunas de lixo não atrapalharam um pouco: elas destruíram o método.

Repare, de passagem, na última linha: a regressão linear, que não tem a menor
condição de representar um seno, ainda assim vai melhor que o KNN com as mesmas 20
colunas. É a figura da Aula 04, §8, outra vez.

O mecanismo é o da Seção 2. A distância euclidiana soma as 20 coordenadas com peso
igual, então 18 delas contribuem só ruído. Os "vizinhos" que o KNN encontra são
vizinhos em ruído, não no que importa.

A floresta aleatória, com as mesmas 20 colunas, chega perto do desempenho de quem
conhecia a resposta. Ela não é mais esperta — ela é **seletiva**: cada divisão
escolhe uma variável, e variáveis que não reduzem impureza raramente são
escolhidas.

In [ ]:
importancias = pd.Series(floresta.feature_importances_,
                         index=[f"x{j+1}" for j in range(d_e)]).sort_values(ascending=False)
print(importancias.head(5).round(4).to_string())
print(f"\nsoma das duas maiores: {importancias.iloc[:2].sum():.3f}")
print(f"soma das outras 18   : {importancias.iloc[2:].sum():.3f}")

fig, ax = subplots(figsize=(5.6, 2.8))
ax.bar(range(d_e), floresta.feature_importances_, color="steelblue")
ax.set_xticks(range(d_e)); ax.set_xticklabels([f"x{j+1}" for j in range(d_e)],
                                              fontsize=6, rotation=90)
ax.set_ylabel("importancia")
ax.set_title("a floresta encontra x1 e x2 sozinha", fontsize=9)

> **A lição.** O KNN não perdeu por ser fraco: perdeu por ser **indiferente à
> estrutura do problema**. Tratar todas as covariáveis igualmente é apostar que
> todas importam igualmente, e quando a aposta está errada ele paga o preço
> integral da dimensão $d$, mesmo que a dimensão efetiva seja $s = 2$.
>
> É a resposta para a Amazon: métodos que **selecionam variáveis** — Lasso (Aula
> 02), árvores, florestas e *boosting* (Aula 06) — operam na dimensão efetiva. O
> KNN, não.

> **Sua vez.** Repita a tabela variando o número de colunas irrelevantes: 0, 2, 5,
> 10, 18. Em que ponto o KNN começa a desmoronar? Compare a curva do KNN com a da
> floresta no mesmo gráfico.

---
## 8. Fuga 2: redundância

Esparsidade e redundância são coisas diferentes, e a diferença decide o método.

Na **esparsidade**, poucas colunas importam e as outras são lixo. Na
**redundância**, todas podem importar, mas elas são tão correlacionadas que os
dados não ocupam o espaço todo: vivem sobre uma subvariedade de dimensão
intrínseca $u \ll d$. Altura, peso e IMC são três variáveis, mas duas determinam a
terceira. Em imagens, *pixels* vizinhos são quase iguais.

E aqui a notícia é boa: sob redundância, o KNN **se adapta sozinho**, porque as
distâncias são medidas dentro da subvariedade. Vamos construir o caso e conferir.

In [ ]:
rng_r = np.random.default_rng(21)
u, d_r, n_r = 2, 20, 2000
M = rng_r.normal(size=(u, d_r))               # mergulho de R^2 em R^20

Z_tr = rng_r.uniform(0, 1, size=(n_r, u))
Z_te = rng_r.uniform(0, 1, size=(3000, u))
Xr_tr, Xr_te = Z_tr @ M, Z_te @ M             # 20 colunas, 2 direcoes de verdade

y_r = np.sin(2 * np.pi * Z_tr[:, 0]) + np.sin(2 * np.pi * Z_tr[:, 1]) \
      + rng_r.normal(0, SIGMA, size=n_r)
r_r_te = np.sin(2 * np.pi * Z_te[:, 0]) + np.sin(2 * np.pi * Z_te[:, 1])

knn_20 = np.mean((KNeighborsRegressor(10).fit(Xr_tr, y_r).predict(Xr_te) - r_r_te) ** 2)
knn_2 = np.mean((KNeighborsRegressor(10).fit(Z_tr, y_r).predict(Z_te) - r_r_te) ** 2)
print(f"KNN nas 20 colunas redundantes : {knn_20:.4f}")
print(f"KNN nas 2 coordenadas latentes : {knn_2:.4f}")
print(f"(para comparar, o caso ESPARSO da Secao 7 dava "
      f"{risco_de(KNeighborsRegressor(10)):.4f} com 20 colunas)")

Vinte colunas, e o KNN vai praticamente tão bem quanto se lhe tivéssemos entregue
as duas coordenadas latentes. Compare com a Seção 7: lá, também com 20 colunas, o
KNN estava perdido. A diferença não está no número de colunas — está em **quantas
direções os dados de fato ocupam**.

E essa quantidade é estimável a partir dos dados, sem saber a resposta. O truque é
a própria Seção 2 ao contrário: se $\rho \approx (k/n)^{1/u}$, então a distância ao
vizinho mais próximo cai como $n^{-1/u}$, e a inclinação de $\log\rho$ contra
$\log n$ é $-1/u$.

In [ ]:
def dimensao_intrinseca(X, tamanhos=(250, 500, 1000, 2000, 4000)):
    tamanhos = [t for t in tamanhos if t <= len(X)]
    raios = []
    for t in tamanhos:
        sub = X[:t]
        dist, _ = NearestNeighbors(n_neighbors=2).fit(sub).kneighbors(sub)
        raios.append(dist[:, 1].mean())          # o [:,0] e o proprio ponto
    inclinacao = np.polyfit(np.log(tamanhos), np.log(raios), 1)[0]
    return -1 / inclinacao


rng_d = np.random.default_rng(21)
casos = {
    "20 colunas independentes": rng_d.uniform(0, 1, size=(4000, 20)),
    "5 colunas independentes": rng_d.uniform(0, 1, size=(4000, 5)),
    "2 colunas independentes": rng_d.uniform(0, 1, size=(4000, 2)),
    "20 colunas sobre um plano (u=2)": rng_d.uniform(0, 1, size=(4000, 2)) @ M,
}
for nome, dados in casos.items():
    print(f"{nome:34s} dimensao estimada: {dimensao_intrinseca(dados):5.1f}")

O estimador reconhece de imediato o plano dentro de $\R^{20}$: dimensão 2. E note
que ele **subestima** quando a dimensão é de fato alta — as 20 colunas
independentes saem como cerca de 15, não 20. Não é bug: com 4000 pontos em 20
dimensões, o regime assintótico de que o estimador depende nem começou. Trate o
número como *"baixa"* ou *"alta"*, não como uma medida precisa.

---
## 9. Fechando o circuito: por que o KNN ganhou na Aula 04

A Aula 04 terminou com um resultado que parecia contradizer tudo isto: no
`superconductivity.csv`, com $d = 81$ colunas, o KNN bateu a Ridge com folga. Com
o que acabamos de montar, dá para explicar — e medir.

In [ ]:
import os

_nome = "superconductivity.csv"
_local = os.path.join("..", "..", "recursos", "dados", _nome)   # repositorio clonado
_url = ("https://raw.githubusercontent.com/HugoCarvalhoUFRJ/ap-maq/"
        "refs/heads/refactoring-baby/recursos/dados/") + _nome  # fallback (ex.: Colab)
_fonte = _local if os.path.exists(_local) else _url

df = pd.read_csv(_fonte)
Xs = df.drop(columns="critical_temp").values
Xs = StandardScaler().fit_transform(Xs)       # senao a escala domina a distancia

rng_s = np.random.default_rng(0)
Xs = Xs[rng_s.permutation(len(Xs))[:8000]]

print(f"colunas do arquivo          : {Xs.shape[1]}")
print(f"dimensao intrinseca estimada: {dimensao_intrinseca(Xs, (500, 1000, 2000, 4000, 8000)):.1f}")

Oitenta e uma colunas, e os dados ocupam um punhado de direções. Faz todo o
sentido quando se sabe o que as colunas são: elas não são 81 medições
independentes, são a média, a média ponderada, a média geométrica, a entropia, a
faixa e o desvio-padrão de um punhado de propriedades atômicas — todas calculadas
a partir da mesma fórmula química. É redundância pura.

Por isso o KNN venceu ali: ele estava, sem saber, trabalhando numa dimensão
efetiva pequena. A regra que fica é a que abre a próxima aula prática:

> o que condena um método de vizinhança não é o número de **colunas**, é o número
> de **direções** que os dados de fato ocupam.

> **Sua vez.** Estime a dimensão intrínseca das 20 colunas do problema **esparso**
> da Seção 7 e compare com as 20 do problema **redundante** da Seção 8. Depois
> explique por que, apesar de as duas situações se chamarem "20 colunas, 2
> importam", só uma delas é detectável pelo estimador de dimensão — e por que isso
> corresponde exatamente a só uma delas ser sobrevivível pelo KNN.

---
## Resumo

| Conceito | Onde apareceu | O que medimos |
|---|---|---|
| $\rho \approx (k/(nV_d))^{1/d}$ | §2 | a fórmula com $V_d$ bate exatamente em $d=1$ e $2$; em $d=20$ a bola é maior que o cubo |
| expoente $1/d$ | §2 | a inclinação em log–log devolve $1/d$; halvar o raio custa dezenas de milhares de vezes mais dados em $d=20$ |
| vizinhança vazia | §3 | em $d=100$ o vizinho mais próximo está a 1/3 do diâmetro |
| contraste de distâncias | §3 | o mais longe deixa de ser mais longe: ordenar por distância vira sorteio |
| taxa do KNN | §4 | inclinação entre $-2/(2+d)$ e $-4/(4+d)$, mais pessimista quanto maior $d$ |
| quase tudo é fronteira | §4 | o miolo do cubo é $(1-2\rho)^d$ — some com $d$ |
| viés² e variância | §5 | as duas parcelas se cruzam no $k^*$; a variância é $\sigma^2/k$ **dado o desenho** |
| tamanho de amostra | §6 | risco $0{,}05$ pede 90 obs. em $d=1$ e $2\times10^{14}$ em $d=20$ |
| esparsidade | §7 | KNN com 20 colunas corta menos da metade do risco; com as 2 certas, 40× melhor; a floresta acha as 2 sozinha |
| redundância | §8 | 20 colunas sobre um plano: o KNN vai tão bem quanto nas 2 latentes |
| dimensão intrínseca | §8–9 | estimável pela inclinação de $\log\rho$ contra $\log n$ |
| o caso real | §9 | 81 colunas do `superconductivity.csv`, pouquíssimas direções |

**Leitura recomendada.** [AME] §4.13 (as contas de taxa do KNN e das séries
ortogonais) e o Capítulo 5 inteiro: §5.1 (maldição e taxas), §5.1.1 (esparsidade),
§5.1.2 (redundância) e §5.2 (KNN e regressão local sob redundância). O Lema da
distância média aos vizinhos é o que a Seção 2 mede; o Teorema do risco, o que a
Seção 4 mede, e a decomposição da Seção 5 é a demonstração dele, medida.

**Para praticar.** `recursos/listas/Lista de exercícios 05.pdf`.

**A seguir.** A Aula 06 apresenta a família que a Seção 7 já deixou vencer sem
apresentação: árvores, florestas e *boosting*. O que elas têm que o KNN não tem é
justamente o que medimos aqui — a capacidade de escolher em quais direções olhar.